# 🏍️ Analisis Statistik & Prediksi Harga Sepeda Motor Bekas
**Regresi Linear Berganda (Multiple Linear Regression) berbasis Python**

analisis statistik terhadap *dataset* penjualan sepeda motor bekas (`motor_second.csv`). Tahapan yang dilakukan meliputi:
1. **Perhitungan Rata-rata (Mean)** untuk melihat karakteristik pasar.
2. **Analisis Korelasi** untuk mengetahui faktor apa yang menaikkan/menurunkan harga.
3. **Pencarian Persamaan Regresi** menggunakan operasi Aljabar Matriks (NumPy).
4. **Simulasi Prediksi** menggunakan antarmuka yang disesuaikan dengan depresiasi nilai saat ini.

In [6]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML

# Memuat dan membersihkan data
df = pd.read_csv('motor_second.csv')
df.columns = df.columns.str.strip()
df = df.dropna()

# Mengambil variabel numerik untuk analisis
df_num = df[['tahun', 'odometer', 'pajak', 'konsumsiBBM', 'mesin', 'harga']]
print("✅ Data berhasil dimuat dan siap dianalisis!")

✅ Data berhasil dimuat dan siap dianalisis!


## 1. Perhitungan Rata-Rata (Mean) 📊
Langkah pertama adalah mencari nilai tengah dari seluruh variabel untuk mengetahui profil umum kendaraan bekas yang beredar di pasaran.

In [7]:
rata_rata = df_num.mean().round(2)
print("=== PROFIL RATA-RATA MOTOR BEKAS ===")
print(f"Tahun Pembuatan : {rata_rata['tahun']} (Umur rata-rata)")
print(f"Jarak Tempuh    : {rata_rata['odometer']:,.0f} km")
print(f"Pajak Kendaraan : Rp {rata_rata['pajak']} Ribu")
print(f"Konsumsi BBM    : {rata_rata['konsumsiBBM']} km/liter")
print(f"Kapasitas Mesin : {rata_rata['mesin']} CC")
print(f"Harga Pasaran   : Rp {rata_rata['harga']:,.2f} (Ribuan)")

=== PROFIL RATA-RATA MOTOR BEKAS ===
Tahun Pembuatan : 2016.61 (Umur rata-rata)
Jarak Tempuh    : 23,522 km
Pajak Kendaraan : Rp 110.76 Ribu
Konsumsi BBM    : 53.55 km/liter
Kapasitas Mesin : 123.91 CC
Harga Pasaran   : Rp 11,359.18 (Ribuan)


## 2. Analisis Korelasi Terhadap Harga 📈
Korelasi (Pearson) digunakan untuk mengukur seberapa kuat pengaruh suatu variabel terhadap harga motor. 
* **Nilai Positif (+):** Berbanding lurus (semakin tinggi nilainya, harga semakin mahal).
* **Nilai Negatif (-):** Berbanding terbalik (semakin tinggi nilainya, harga semakin murah).

In [8]:
korelasi = df_num.corr()['harga'].drop('harga').round(4)
print("=== TINGKAT KORELASI VARIABEL TERHADAP HARGA ===")
for indeks, nilai in korelasi.items():
    sifat = "Positif (Menaikkan Harga)" if nilai > 0 else "Negatif (Menurunkan Harga)"
    print(f"- {indeks.ljust(12)}: {nilai:7.4f}  --> {sifat}")

=== TINGKAT KORELASI VARIABEL TERHADAP HARGA ===
- tahun       :  0.5703  --> Positif (Menaikkan Harga)
- odometer    : -0.4245  --> Negatif (Menurunkan Harga)
- pajak       :  0.3624  --> Positif (Menaikkan Harga)
- konsumsiBBM : -0.4564  --> Negatif (Menurunkan Harga)
- mesin       :  0.3961  --> Positif (Menaikkan Harga)


## 3. Persamaan Regresi Linear Berganda 🧮
Karena terdapat lebih dari satu variabel *X*, pemodelan ini menggunakan rumus regresi linier berganda:
$$Y = \beta_0 + \beta_1X_1 + \beta_2X_2 + \beta_3X_3 + \beta_4X_4 + \beta_5X_5$$

Untuk mencari bobot Koefisien ($\beta$) secara komputasi, digunakan rumus invers matriks Aljabar Linear (Normal Equation) yang logikanya setara dengan fungsi `=MINVERSE` dan `=MMULT` pada Microsoft Excel:
$$\beta = (X^T X)^{-1} X^T Y$$

In [9]:
# Pemisahan X dan Y
X = df_num[['tahun', 'odometer', 'pajak', 'konsumsiBBM', 'mesin']].values
Y = df_num['harga'].values

# Menambahkan konstanta 1 untuk intercept (b0)
X_matriks = np.column_stack((np.ones(X.shape[0]), X))

# Perhitungan Matriks NumPy
X_T = X_matriks.T
Invers_XT_X = np.linalg.inv(X_T.dot(X_matriks))
XT_Y = X_T.dot(Y)

# Hasil Bobot Koefisien
bobot = Invers_XT_X.dot(XT_Y)
b0, b1, b2, b3, b4, b5 = bobot

print("=== HASIL MATRIKS KOEFISIEN REGRESI ===")
print(f"b0 (Konstanta) : {b0:,.2f}")
print(f"b1 (Tahun)     : {b1:,.2f}")
print(f"b2 (Odometer)  : {b2:,.4f}")
print(f"b3 (Pajak)     : {b3:,.2f}")
print(f"b4 (BBM)       : {b4:,.2f}")
print(f"b5 (Mesin)     : {b5:,.2f}")

=== HASIL MATRIKS KOEFISIEN REGRESI ===
b0 (Konstanta) : -2,134,301.20
b1 (Tahun)     : 1,060.13
b2 (Odometer)  : -0.0541
b3 (Pajak)     : 3.90
b4 (BBM)       : -72.26
b5 (Mesin)     : 100.81


## 4. Simulasi Prediksi & Analisis Depresiasi 🎯
Berdasarkan matriks koefisien yang didapatkan, sistem ini telah siap memprediksi harga kendaraan. Namun, ada satu hal penting dari model yang kita bangun, yaitu perhitungan **Depresiasi (Penyusutan Harga)**.

**💡 Analisis: Dari mana asal angka penyusutan Rp 1,6 Juta/tahun?**
Algoritma regresi secara otomatis melacak bahwa penyusutan harga motor disebabkan oleh kombinasi dua faktor utama:
1. **Penyusutan Umur (b1 = 1060.13):** Secara konstan, setiap bertambah umur 1 tahun, harga motor menyusut sekitar **Rp 1.060.000**.
2. **Penyusutan Jarak Tempuh (b2 = -0.0541):** Setiap dipakai jalan 1 km, harga turun sebesar Rp 54,1. Jika diasumsikan pemakaian normal motor komuter adalah 10.000 km/tahun, maka penyusutannya sekitar **Rp 541.000**.

Jika digabungkan, total depresiasi per tahun adalah: Rp 1.060.000 + Rp 541.000 = **Rp 1.601.000 (Dibulatkan Rp 1,6 Juta per tahun)**.

> **⚠️ Koreksi Waktu (Time-Series Adjustment):** > Karena dataset sampel (`motor_second.csv`) merekam kondisi harga pasar di sekitar tahun 2024, maka untuk memprediksi harga di tahun ini (selisih 2 tahun), sistem simulasi di bawah ini telah diprogram untuk **memotong hasil prediksi sebesar Rp 3,2 Juta**.

In [10]:
def prediksi_realtime(tahun, odometer, pajak, bbm, mesin):
    # 1. Hitung Rumus Matematika
    harga_awal = b0 + (b1 * tahun) + (b2 * odometer) + (b3 * pajak) + (b4 * bbm) + (b5 * mesin)
    
    # 2. Koreksi Waktu (Depresiasi)
    harga_final = max(0, (harga_awal - 3200) * 1000)
    
    # 3. Desain UI HTML/CSS
    ui = f"""
    <div style="font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; 
                background: linear-gradient(145deg, #ffffff, #f0f3f5); 
                padding: 30px; 
                border-radius: 20px; 
                box-shadow: 10px 10px 20px #d1d9e6, -10px -10px 20px #ffffff; 
                width: 480px; 
                text-align: center;
                border: 1px solid #e1e5ee;
                margin-top: 10px;">
        
        <div style="background-color: #e3f2fd; color: #1565c0; padding: 6px 16px; 
                    border-radius: 20px; display: inline-block; font-size: 12px; 
                    font-weight: bold; letter-spacing: 1px; margin-bottom: 15px;">
            🤖 PRICE ESTIMATOR
        </div>
        
        <h2 style="color: #2c3e50; margin: 0; font-size: 18px; font-weight: 600;">
            Taksiran Nilai Pasar Kendaraan
        </h2>
        
        <h1 style="color: #27ae60; font-size: 48px; margin: 15px 0 5px 0; 
                   font-weight: 800; letter-spacing: -1px; text-shadow: 1px 1px 0px rgba(0,0,0,0.05);">
            Rp {harga_final:,.0f}
        </h1>
        
        <div style="border-top: 2px dashed #cbd5e1; width: 80%; margin: 25px auto 20px auto;"></div>
        
        <div style="background-color: #fff3cd; color: #856404; padding: 12px 15px; 
                    border-radius: 10px; font-size: 12px; text-align: left; line-height: 1.5;
                    border-left: 4px solid #ffc107;">
            <strong>⚠️ Info Depresiasi Aktif:</strong><br>
            Harga di atas telah dipotong otomatis sebesar <b>Rp 3.200.000</b> dari standar model awal untuk menyesuaikan penyusutan umur & jarak tempuh di tahun ini.
        </div>
    </div>
    <br>
    """
    # Tampilkan UI ke layar
    display(HTML(ui))

# TOMBOL SLIDER
# Menyesuaikan ukuran slider
style_widget = {'description_width': 'initial'}
layout_widget = widgets.Layout(width='500px')

# Menampilkan Tuas Input
widgets.interact(prediksi_realtime, 
                 tahun=widgets.IntSlider(min=2010, max=2024, step=1, value=2018, description='📅 Tahun Rilis:', style=style_widget, layout=layout_widget),
                 odometer=widgets.IntSlider(min=0, max=100000, step=1000, value=15000, description='🛣️ Jarak (KM):', style=style_widget, layout=layout_widget),
                 pajak=widgets.IntSlider(min=50, max=500, step=10, value=150, description='🧾 Pajak (Rb):', style=style_widget, layout=layout_widget),
                 bbm=widgets.IntSlider(min=30, max=80, step=1, value=45, description='⛽ BBM (km/l):', style=style_widget, layout=layout_widget),
                 mesin=widgets.Dropdown(options=[110, 125, 150, 155, 250], value=125, description='⚙️ CC Mesin:', style=style_widget, layout=layout_widget));

interactive(children=(IntSlider(value=2018, description='📅 Tahun Rilis:', layout=Layout(width='500px'), max=20…